In [0]:
dbutils.widgets.text("target_catalog", "tpch_dev", "Target Catalog")
target_catalog = dbutils.widgets.get("target_catalog")
print(f"Processing silver layer for catalog: {target_catalog}")

In [0]:
from pyspark.sql import functions as F

bronze_orders = spark.table(f"{target_catalog}.bronze.orders")

# Simulate: take a sample of orders as if they arrived again in a later batch
# with a newer ingestion timestamp (common real-world dedup scenario)
duplicate_batch = (
    bronze_orders
    .sample(fraction=0.05, seed=42)
    .withColumn("_ingested_at", F.current_timestamp())
)

original_batch = bronze_orders.withColumn(
    "_ingested_at", F.lit("2024-01-01T00:00:00").cast("timestamp")
)

# Combine: this represents bronze accumulating multiple ingestion runs
orders_with_duplicates = original_batch.unionByName(duplicate_batch)

print(f"Original rows: {original_batch.count()}")
print(f"After adding duplicate batch: {orders_with_duplicates.count()}")

In [0]:
from pyspark.sql.window import Window

window_spec = Window.partitionBy("o_orderkey").orderBy(F.col("_ingested_at").desc())

deduped_orders = (
    orders_with_duplicates
    .withColumn("_row_num", F.row_number().over(window_spec))
    .filter(F.col("_row_num") == 1)
    .drop("_row_num")
)

print(f"After dedup: {deduped_orders.count()}")  # should match original count

In [0]:
from delta.tables import DeltaTable

silver_orders_table = f"{target_catalog}.silver.orders"

# Create table if it doesn't exist yet
if not spark.catalog.tableExists(silver_orders_table):
    deduped_orders.write.format("delta").saveAsTable(silver_orders_table)
    print(f"Created {silver_orders_table}")
else:
    delta_table = DeltaTable.forName(spark, silver_orders_table)
    (
        delta_table.alias("target")
        .merge(
            deduped_orders.alias("source"),
            "target.o_orderkey = source.o_orderkey"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print(f"Merged into {silver_orders_table}")

In [0]:
def merge_to_silver(table_name, key_column, target_catalog):
    bronze_df = spark.table(f"{target_catalog}.bronze.{table_name}")
    silver_table = f"{target_catalog}.silver.{table_name}"

    if not spark.catalog.tableExists(silver_table):
        bronze_df.write.format("delta").saveAsTable(silver_table)
        print(f"Created {silver_table}: {bronze_df.count()} rows")
    else:
        delta_table = DeltaTable.forName(spark, silver_table)
        (
            delta_table.alias("target")
            .merge(
                bronze_df.alias("source"),
                f"target.{key_column} = source.{key_column}"
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
        print(f"Merged into {silver_table}: {bronze_df.count()} source rows")

In [0]:
def merge_to_silver_composite(table_name, key_columns, target_catalog):
    bronze_df = spark.table(f"{target_catalog}.bronze.{table_name}")
    silver_table = f"{target_catalog}.silver.{table_name}"

    merge_condition = " AND ".join([f"target.{k} = source.{k}" for k in key_columns])

    if not spark.catalog.tableExists(silver_table):
        bronze_df.write.format("delta").saveAsTable(silver_table)
        print(f"Created {silver_table}: {bronze_df.count()} rows")
    else:
        delta_table = DeltaTable.forName(spark, silver_table)
        (
            delta_table.alias("target")
            .merge(bronze_df.alias("source"), merge_condition)
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
        print(f"Merged into {silver_table}: {bronze_df.count()} source rows")

In [0]:
merge_to_silver_composite("lineitem", ["l_orderkey", "l_linenumber"], target_catalog)

In [0]:
merge_to_silver("supplier", "s_suppkey", target_catalog)
merge_to_silver("part", "p_partkey", target_catalog)

In [0]:
merge_to_silver("customer", "c_custkey", target_catalog)
merge_to_silver("nation", "n_nationkey", target_catalog)
merge_to_silver("region", "r_regionkey", target_catalog)
merge_to_silver_composite("lineitem", ["l_orderkey", "l_linenumber"], target_catalog)
merge_to_silver("supplier", "s_suppkey", target_catalog)
merge_to_silver("part", "p_partkey", target_catalog)

In [0]:
spark.sql(f"SHOW TABLES IN {target_catalog}.silver").display()
